# Phase 2 - V2 BERT NER on Classified Positives

**Purpose**: Extract entities from Phase 2 classified positive papers using V2 BERT NER

**Script**: `scripts/07a_run_phase2_v2_ner.py`

**Environment**: Google Colab with GPU support

**Runtime**: ~30-60 minutes (GPU) | ~3-5 hours (CPU)

**Note**: Requires classification results from 06a (V2) or 06b (PyCaret)

---

## Usage

1. **Prerequisites**: Run `phase2_v2_classification_150k_VB.ipynb` OR `phase2_pycaret_classification_150k_VB.ipynb` first
2. **Runtime**: Select GPU runtime (Runtime → Change runtime type → GPU)
3. **Execute cells**: Run cells in order from top to bottom
4. **Monitor**: Watch progress in cell outputs
5. **Results**: Automatically saved to Google Drive

---

In [ ]:
# =============================================================================
# CELL 1: MOUNT GOOGLE DRIVE
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully")
print("📦 Phase 2 files are now accessible")
print("🔗 Ready for V2 BERT NER on classified positives")

In [ ]:
# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================
import os
from datetime import datetime

# =============================================================================
# MODE SELECTION
# =============================================================================
TEST_MODE = False  # Set to True for quick testing (~5 min with first 100 positives)
                   # Set to False for full run (~30-60 min with all positives)

# Session Management (IMPORTANT: Use same SESSION_ID as classification run!)
import random
import string

# OPTION 1: Use session ID from previous classification run
# Uncomment and paste your classification SESSION_ID here:
# SESSION_ID = "2025-11-14-abc123"  # <-- PASTE YOUR SESSION_ID HERE

# OPTION 2: Generate new session ID (will look for any classification results)
SESSION_ID = f"{datetime.now().strftime('%Y-%m-%d')}-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}"

TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

print(f"\n🔖 Session ID: {SESSION_ID}")
print(f"📅 Timestamp: {TIMESTAMP}")

# =============================================================================
# PATH CONFIGURATION
# =============================================================================
BASE_PATH = '/content/drive/MyDrive'
REPO_NAME = 'inventory_2022'
REPO_PATH = os.path.join(BASE_PATH, REPO_NAME)

if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(f"❌ Repository not found at: {REPO_PATH}")

VALIDATION_PATH = os.path.join(REPO_PATH, 'validation_spacy_v_BERT')

if not os.path.exists(VALIDATION_PATH):
    raise FileNotFoundError(f"❌ Validation directory not found at: {VALIDATION_PATH}")

print(f"✓ Repository found: {REPO_PATH}")
print(f"✓ Validation directory: {VALIDATION_PATH}")

os.chdir(VALIDATION_PATH)

os.environ['TEST_MODE'] = str(TEST_MODE)
os.environ['SESSION_ID'] = SESSION_ID

if TEST_MODE:
    print("\n🧪 TEST MODE ENABLED")
    print("   ⏱️  Expected runtime: ~5 minutes")
    print("   📊 Dataset: First 100 classified positives")
else:
    print("\n🚀 PRODUCTION MODE ENABLED")
    print("   ⏱️  Expected runtime: ~30-60 minutes")
    print("   📊 Dataset: All classified positives from Phase 2")

print(f"\n📁 Working directory: {os.getcwd()}")
print("\n" + "="*60)
print("CONFIGURATION LOADED")
print("="*60)

In [ ]:
# =============================================================================
# CELL 3: INSTALL DEPENDENCIES
# =============================================================================

print("🔧 Installing packages...")
!pip install transformers datasets evaluate seqeval nltk -q

print("\n✅ Dependencies installed")

import nltk
import ssl

print("Downloading NLTK data...")
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab', quiet=True)
print("✅ NLTK data downloaded")

print("\n✅ Environment setup complete")

In [ ]:
# =============================================================================
# CELL 4: GPU CHECK
# =============================================================================

import torch

print("🔍 GPU Environment Check:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print("✅ GPU acceleration available")
    print("\n⚡ NER will run ~5-10x faster on GPU")
else:
    print("⚠️ No GPU detected - NER will be slow")
    print("Consider enabling GPU runtime: Runtime > Change runtime type > GPU")
    print("\n⏱️  Expected runtime on CPU: ~3-5 hours")

In [ ]:
# =============================================================================
# CELL 5: VERIFY INPUT FILES
# =============================================================================

import sys
from pathlib import Path

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

src_path = str(Path(REPO_PATH) / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("🔍 Verifying input files...\n")

# Check for classification results (will auto-discover V2 or PyCaret)
classif_dir = Path(VALIDATION_PATH) / "results/phase2/classification"
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ""

classif_files = list(classif_dir.glob("*classification_150k*.csv"))

if classif_files:
    print(f"✓ Found {len(classif_files)} classification result file(s):")
    for f in sorted(classif_files, key=lambda x: x.stat().st_mtime, reverse=True)[:3]:
        print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")
    print(f"\n  Script will use most recent matching your SESSION_ID")
else:
    raise FileNotFoundError(f"❌ No classification results found in {classif_dir}")
    print("Please run phase2_v2_classification_150k_VB.ipynb OR phase2_pycaret_classification_150k_VB.ipynb first")

# Check V2 NER model
model_path = Path(REPO_PATH) / "out/original_model/named_entity_recognition.pt"

if model_path.exists():
    print(f"\n✓ V2 NER model: {model_path}")
    print(f"  Size: {model_path.stat().st_size / 1024 / 1024:.1f} MB")
else:
    raise FileNotFoundError(f"Required model not found: {model_path}")

# Check script
script_path = Path(VALIDATION_PATH) / "scripts/07a_run_phase2_v2_ner.py"
if not script_path.exists():
    raise FileNotFoundError(f"Required script not found: {script_path}")
print(f"\n✓ Script verified: {script_path}")

print("\n✅ All required files verified")

In [ ]:
# =============================================================================
# CELL 6: RUN V2 BERT NER
# =============================================================================

from pathlib import Path

print("🚀 Running V2 BERT NER on classified positives...\n")
print("="*70)

script_path = Path(VALIDATION_PATH) / "scripts/07a_run_phase2_v2_ner.py"
%run {script_path}

print("\n" + "="*70)
print("✅ V2 BERT NER Complete")

In [ ]:
# =============================================================================
# CELL 7: DISPLAY RESULTS
# =============================================================================
import pandas as pd
from pathlib import Path

TEST_MODE = os.environ.get('TEST_MODE', 'False').lower() == 'true'
SESSION_ID = os.environ.get('SESSION_ID', '')
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ("_test" if TEST_MODE else "")

print("📊 NER Results Summary\n")
print("="*70)

results_file = Path(VALIDATION_PATH) / f"results/phase2/ner/v2_ner_150k{output_suffix}.csv"

if results_file.exists():
    df_results = pd.read_csv(results_file)
    
    print(f"\n📄 Results file: {results_file}")
    print(f"📦 File size: {results_file.stat().st_size / 1024:.1f} KB")
    
    print(f"\n📊 Entity Extraction Statistics:")
    print(f"   Total entities: {len(df_results):,}")
    print(f"   Unique papers: {df_results['ID'].nunique():,}")
    print(f"   Avg entities/paper: {len(df_results)/df_results['ID'].nunique():.2f}")
    
    if 'label' in df_results.columns:
        print(f"\n📈 Entity Types:")
        for label, count in df_results['label'].value_counts().items():
            print(f"   - {label}: {count:,} ({100*count/len(df_results):.1f}%)")
    
    print(f"\n📋 Sample Entities (first 10):")
    display_cols = ['ID', 'text']
    if 'label' in df_results.columns:
        display_cols.append('label')
    
    print(df_results[display_cols].head(10))
    
    print(f"\n✅ Results successfully saved to Drive")
    print(f"   Location: {results_file}")
    
else:
    print(f"⚠️ Results file not found: {results_file}")

print("\n" + "="*70)

In [ ]:
# =============================================================================
# CELL 8: COMPLETION SUMMARY
# =============================================================================

from datetime import datetime
from pathlib import Path

TEST_MODE = os.environ.get('TEST_MODE', 'False').lower() == 'true'
SESSION_ID = os.environ.get('SESSION_ID', '')
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ("_test" if TEST_MODE else "")
results_file = Path(VALIDATION_PATH) / f"results/phase2/ner/v2_ner_150k{output_suffix}.csv"

print("\n" + "🎉" * 35)
print("🎉 V2 BERT NER COMPLETE")
print("🎉" * 35)

print(f"\n📊 Session Information:")
print(f"   Session ID: {SESSION_ID if SESSION_ID else 'N/A'}")
print(f"   Mode: {'TEST' if TEST_MODE else 'PRODUCTION'}")
print(f"   Completion Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📁 Output Files:")
if results_file.exists():
    print(f"   📄 {results_file}")
    df_check = pd.read_csv(results_file)
    print(f"   📊 {len(df_check):,} entities from {df_check['ID'].nunique():,} papers")
else:
    print(f"   ⚠️ Results file not created (check errors above)")

print(f"\n🎯 Next Steps:")
print(f"   1. Review extracted entities above")
print(f"   2. Optional: Run spaCy Hybrid NER for comparison (phase2_spacy_ner_150k_VB.ipynb)")
print(f"   3. Analyze entity types and frequencies")

print(f"\n🏁 Notebook execution complete!")
print(f"📊 Results are automatically saved to Google Drive")